# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsse23-060/Saram_Flyrank1/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os

# Read the token from the environment or a private Colab Secret named HF_TOKEN.
# Never paste a token into notebook source: this repository is public.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
assert HF_TOKEN, 'Add a Colab Secret named HF_TOKEN, enable notebook access, and rerun.'

from huggingface_hub import HfApi, hf_hub_url, get_hf_file_metadata
api = HfApi(token=HF_TOKEN)
print('Authenticated Hugging Face account:', api.whoami()['name'])
api.dataset_info('FlyRank/internship-warehouse', token=HF_TOKEN)
test_url = hf_hub_url(
    repo_id='FlyRank/internship-warehouse',
    filename='fact_content_daily_performance/month=2025-03/data_0.parquet',
    repo_type='dataset',
)
metadata = get_hf_file_metadata(test_url, token=HF_TOKEN)
print(f'Daily partition access confirmed ({metadata.size:,} bytes).')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
import duckdb

# Cache only the tables and months used below. The cache is outside the repo,
# so no dataset can be committed accidentally.
IN_COLAB = Path('/content').exists()
cache_dir = (
    Path('/content/flyrank_warehouse_cache')
    if IN_COLAB else Path.home() / '.cache' / 'flyrank_warehouse_cache'
)
cache_root = Path(snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir=str(cache_dir),
    allow_patterns=[
        'dim_clients.parquet',
        'dim_content.parquet',
        'fact_content_query_90d.parquet',
        'fact_content_daily_performance/month=2026-04/*.parquet',
        'fact_content_daily_performance/month=2026-05/*.parquet',
        'fact_content_daily_performance/month=2026-06/*.parquet',
    ],
    max_workers=8,
))

daily_files = sorted(
    cache_root.glob('fact_content_daily_performance/month=2026-0[4-6]/*.parquet')
)
assert daily_files, 'No April-June daily partitions were cached.'

def sql_path(path):
    return path.resolve().as_posix().replace("'", "''")

daily_list = ', '.join(f"'{sql_path(path)}'" for path in daily_files)
con = duckdb.connect()
TABLES = {
    'dim_clients': f"read_parquet('{sql_path(cache_root / 'dim_clients.parquet')}')",
    'dim_content': f"read_parquet('{sql_path(cache_root / 'dim_content.parquet')}')",
    'fact_daily': f"read_parquet([{daily_list}], hive_partitioning=true)",
    'fact_query_90d': f"read_parquet('{sql_path(cache_root / 'fact_content_query_90d.parquet')}')",
}

print(f'Cached {len(daily_files):,} daily Parquet files outside the repository.')
for name, src in TABLES.items():
    n = con.execute(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date > DATE '2026-05-31'
                    THEN gsc_impressions ELSE 0 END) AS imp_last30,
           SUM(CASE WHEN report_date > DATE '2026-05-01'
                         AND report_date <= DATE '2026-05-31'
                    THEN gsc_impressions ELSE 0 END) AS imp_prev30,
           SUM(CASE WHEN report_date > DATE '2026-05-31'
                    THEN gsc_clicks ELSE 0 END) AS clk_last30,
           AVG(CASE WHEN report_date > DATE '2026-05-01'
                        AND report_date <= DATE '2026-05-31'
                        AND gsc_avg_position > 0
                    THEN gsc_avg_position END) AS pos_prev30,
           STDDEV_SAMP(CASE WHEN report_date > DATE '2026-05-01'
                                  AND report_date <= DATE '2026-05-31'
                                  AND gsc_avg_position > 0
                            THEN gsc_avg_position END) AS pos_prev30_volatility
    FROM {TABLES['fact_daily']}
    WHERE report_date > DATE '2026-05-01'
      AND report_date <= DATE '2026-06-30'
    GROUP BY 1, 2
    HAVING imp_prev30 >= 100
""").df()

assert features.duplicated(['client_hash_id', 'content_hash_id']).sum() == 0
print(f'{len(features):,} content items with enough history')
features.head()


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
